In [ ]:
import os
import qdrant_client
from dotenv import load_dotenv
from llama_index.core import VectorStoreIndex, Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.core.vector_stores.types import VectorStoreQueryMode
load_dotenv()

# Qdrant Client
url_qdrant = os.getenv("QDRANT_URL")
client = qdrant_client.QdrantClient(url=url_qdrant)

url_vllm = os.getenv("VLLM_API_BASE_URL")

embed_model = OpenAIEmbedding(
    api_base=url_vllm,
    model_name="BAAI/bge-m3",
    api_key="null",
)

Settings.embed_model = embed_model


vector_store = QdrantVectorStore(
    client=client,
    collection_name="WAMASRAG",
    enable_hybrid=True,
    dense_vector_name="bge_m3",
    sparse_vector_name="bm25",
    fastembed_sparse_model="Qdrant/bm25"
)



#IMPORTANTISSIMO
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)



def search_knowledge_base(query_text: str):
    print(f"Ricerca in corso per: '{query_text}'...")
    
    # Configura il retriever
    retriever = index.as_retriever(
        vector_store_query_mode=VectorStoreQueryMode.HYBRID,
        similarity_top_k=10, 
        sparse_top_k=10,    
        alpha=0.5           
    )
    
    
    results = retriever.retrieve(query_text)
    
    return results



if __name__ == "__main__":
    user_question = "come faccio per arrestare una baia di picking?"
    
    nodes = search_knowledge_base(user_question)
    
    print(f"\nTrovati {len(nodes)} risultati:\n")
    for i, node_ws in enumerate(nodes, 1):
        score = node_ws.score
        filename = node_ws.node.metadata.get('origin_filename', 'N/A')
        page = node_ws.node.metadata.get('pages', 'N/A')
        content = node_ws.node.get_content()#.replace("\n", " ")
        
        print(f"{i}. [Score: {score:.3f}] - File: {filename} (Pag. {page})")
        print(f"   text: {content}...\n")

Ricerca in corso per: 'come faccio per arrestare una baia di picking?'...

Trovati 10 risultati:

1. [Score: 0.987] - File: UTL-MAN_Arresto_baie_picking.pdf (Pag. [2])
   text: [CONTEXT
Il documento descrive la procedura per l'arresto delle baie di picking in un sistema di gestione logistica, con l'obiettivo di permettere il regolare svolgimento di attività sulla stazione stessa. La procedura fornisce istruzioni dettagliate su come arrestare una baia, accelerare l'arresto e ripristinare gli ordini cancellati.]
3. Come si arresta una baia di picking
Prima di eseguire un'operazione sulla baia di picking , bisogna assicurarsi che questa sia in stato di ARRESTO. Se così non fosse, non sarebbe possibile eseguire nessuna attività sulla suddetta baia.
Per arrestare una baia, ci si connette alla pagina di WAMAS WH083 e si preme 'cerca' senza inserire nessun parametro di ricerca per mostrare l'elenco completo delle baie....

2. [Score: 0.898] - File: UTL-MAN_Arresto_baie_picking.pdf (Pag. [2])
